# Music Annotation Project V8.7
## Notebook 51R2: Frozen M3 vs M5-REMOVE vs diagnostic M4 Reconstruction Crossover

**Development-only. No production or promotion authority.**

This replacement corrects the previous Notebook 51 lineage/model-loading errors.

### Fixed experiment
- Mask arms: **M3**, **M5_REMOVE**, **M4_DIAGNOSTIC**.
- `M5_REMOVE = M3 minus Notebook 50 SAFE_REMOVE`, with **no SAFE_PROTECT intervention**.
- Restorers: **R1_biharmonic**, **R2_telea**, **R3_learned_unet**.
- R3 uses Notebook 46R6's exact `TinyUNet` architecture and the checkpoint for the page's held-out `source_group`.
- M3 and M4 whole-page candidates are reused from the completed Notebook 46R6 run where available. This prevents accidental restorer drift.
- M5-REMOVE is newly reconstructed under the same frozen restorer implementations.
- M4 remains diagnostic only.
- Clean truth is unavailable until the complete candidate inventory is written and hashed.
- Outside the authoritative mask is byte-for-byte immutable.
- Paired page and source-group analyses are mandatory.
- A secondary endpoint tests whether the between-restorer gap contracts from M3 to M5-REMOVE.

**Important:** Notebook 46R6 does not create `experiment_identity.json`. This notebook instead verifies its actual durable identity set: `source_record.json`, `restorer_pre_registration.json`, `v8_6_verdict.json`, `audit/output_completion.json`, and `audit/learned_model_inventory.csv`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow scikit-image matplotlib torch torchvision


## 1. Configuration, output tree and pre-registration


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, math, random, shutil, sys, platform, zipfile, gc, time, os
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from skimage.restoration import inpaint as sk_inpaint
from skimage.metrics import structural_similarity as ssim
from torch import nn

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
N45_ROOT=ROOT/'observed_print_classifier_v8_6_notebook45'
N46_ROOT=ROOT/'observed_print_classifier_v8_6_notebook46'
N50_ROOT=ROOT/'observed_print_classifier_v8_7_notebook50'
OUTROOT=ROOT/'observed_print_classifier_v8_7_notebook51'

N45_RUN_OVERRIDE=None
N46_RUN_OVERRIDE=None
N50_RUN_OVERRIDE=None
RESUME_RUN_OVERRIDE=None
AUTO_RESUME_LATEST_INCOMPLETE=True
ALLOW_NEW_RUN_IF_NONE_FOUND=True

SEED=51002
BOOT=10000
PATCH=128
CROP_MARGIN=48
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE!='cuda': raise RuntimeError('Notebook 51 R3 learned restorer requires a GPU runtime.')

ARMS=['M3','M5_REMOVE','M4_DIAGNOSTIC']
RESTORERS=['R1_biharmonic','R2_telea','R3_learned_unet']
PRIMARY_RESTORER='R3_learned_unet'


def sha(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()

def array_hash(a):
    a=np.ascontiguousarray(a); h=hashlib.sha256(); h.update(str(a.dtype).encode()); h.update(str(a.shape).encode()); h.update(a.tobytes()); return h.hexdigest()

def atomic_json(path,value):
    path=Path(path); tmp=Path(str(path)+'.tmp'); tmp.write_text(json.dumps(value,indent=2,sort_keys=True,default=str)); tmp.replace(path)

def atomic_csv(frame,path):
    path=Path(path); tmp=Path(str(path)+'.tmp'); frame.to_csv(tmp,index=False); tmp.replace(path)

def require(condition,message):
    if not condition:
        if 'AUDIT' in globals(): atomic_json(AUDIT/'fatal_failure.json',{'utc':datetime.now(timezone.utc).isoformat(),'message':message})
        raise RuntimeError(message)

def newest_complete(root,required):
    hits=[]
    if Path(root).exists():
        for p in Path(root).iterdir():
            if p.is_dir() and all((p/r).exists() for r in required): hits.append(p)
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None

# Resume only Notebook51R2 directories carrying this pre-registration.
def resumable_runs(root):
    hits=[]
    if Path(root).exists():
        for p in Path(root).iterdir():
            if p.is_dir() and (p/'01_pre_registration.json').exists():
                try:
                    pre=json.loads((p/'01_pre_registration.json').read_text())
                    if pre.get('version')=='8.7-notebook51R2': hits.append(p)
                except Exception: pass
    return sorted(hits,key=lambda p:p.stat().st_mtime,reverse=True)

if RESUME_RUN_OVERRIDE:
    OUT=Path(RESUME_RUN_OVERRIDE); RESUME=True
elif AUTO_RESUME_LATEST_INCOMPLETE:
    rr=resumable_runs(OUTROOT); OUT=rr[0] if rr else None; RESUME=OUT is not None
else:
    OUT=None; RESUME=False
if OUT is None:
    require(ALLOW_NEW_RUN_IF_NONE_FOUND,'No resumable Notebook51R2 run found.')
    OUT=OUTROOT/(datetime.now().strftime('%Y%m%d_%H%M%S')+'_v8_7_notebook51R2'); RESUME=False

INPUT=OUT/'01_source_lock'; CHECK=OUT/'02_page_checkpoints'; CAND=OUT/'03_candidates'; LOCK=OUT/'04_candidate_lock'; TABLES=OUT/'05_tables'; VIS=OUT/'06_visual_audit'; AUDIT=OUT/'07_audit'; UPLOAD=OUT/'upload'
for p in [OUT,INPUT,CHECK,CAND,LOCK,TABLES,VIS,AUDIT,UPLOAD]: p.mkdir(parents=True,exist_ok=True)

PREREG={'version':'8.7-notebook51R2','development_only':True,'mask_arms':ARMS,'restorers':RESTORERS,'primary_restorer':PRIMARY_RESTORER,'m5_definition':'M3 minus Notebook50 SAFE_REMOVE only','safe_protect_intervention':False,'m4_role':'diagnostic only','truth_firewall':'clean truth loaded only after candidate lock','bootstrap_resamples':BOOT,'promotion_authority':False}
pre_path=OUT/'01_pre_registration.json'
if pre_path.exists(): require(json.loads(pre_path.read_text())==PREREG,'Resume blocked: Notebook51R2 pre-registration changed.')
else: atomic_json(pre_path,PREREG)
print('Mode:', 'RESUME' if RESUME else 'NEW RUN'); print('Output:',OUT); print('GPU:',torch.cuda.get_device_name(0))


## 2. Resolve exact completed Notebook 45, 46R6 and 50 runs


In [ ]:
REQ45=['mask_lock.json','source_record.json','tables/region_manifest.parquet']
REQ46=['source_record.json','restorer_pre_registration.json','v8_6_verdict.json','audit/output_completion.json','audit/learned_model_inventory.csv','tables/whole_page_candidate_inventory.csv']
REQ50=['notebook50_verdict.json','m5_map_inventory.csv']

N45=Path(N45_RUN_OVERRIDE) if N45_RUN_OVERRIDE else newest_complete(N45_ROOT,REQ45)
N46=Path(N46_RUN_OVERRIDE) if N46_RUN_OVERRIDE else newest_complete(N46_ROOT,REQ46)
N50=Path(N50_RUN_OVERRIDE) if N50_RUN_OVERRIDE else newest_complete(N50_ROOT,REQ50)
require(N45 is not None,'No complete Notebook45 run found. Set N45_RUN_OVERRIDE.')
require(N46 is not None,'No complete Notebook46R6 run found. Set N46_RUN_OVERRIDE.')
require(N50 is not None,'No complete Notebook50 run found. Set N50_RUN_OVERRIDE.')

lock45=json.loads((N45/'mask_lock.json').read_text())
source45=json.loads((N45/'source_record.json').read_text())
source46=json.loads((N46/'source_record.json').read_text())
pre46=json.loads((N46/'restorer_pre_registration.json').read_text())
verdict46=json.loads((N46/'v8_6_verdict.json').read_text())
complete46=json.loads((N46/'audit/output_completion.json').read_text())
verdict50=json.loads((N50/'notebook50_verdict.json').read_text())

require(verdict46.get('valid_run') is True,'Notebook46R6 verdict is not valid_run=True.')
require(complete46.get('complete') is True,'Notebook46R6 output_completion is not complete.')
require('R3_learned_unet' in pre46.get('restorers',[]),'Notebook46R6 pre-registration lacks R3_learned_unet.')
require(pre46.get('mask_lock_id')==source46.get('mask_lock_id')==lock45.get('lock_id'),'Notebook45/46 mask-lock lineage mismatch.')
require(verdict50.get('verdict')=='M5_REMOVE_ONLY_READY_FOR_NOTEBOOK51_CROSSOVER','Notebook50 verdict does not authorise REMOVE-only crossover.')

regions=pd.read_parquet(N45/'tables/region_manifest.parquet')
need={'page','source_group','region_id','y0','y1','x0','x1'}
require(need.issubset(regions.columns),f'Notebook45 region manifest missing {sorted(need-set(regions.columns))}')
regions['page']=regions.page.astype(str); regions['source_group']=regions.source_group.astype(str)
require(not regions[['page','region_id']].duplicated().any(),'Duplicate Notebook45 region identity.')

source_record={'notebook45':str(N45),'notebook46':str(N46),'notebook50':str(N50),'mask_lock_id':lock45.get('lock_id'),'notebook46_verdict':verdict46.get('decision'),'notebook50_verdict':verdict50.get('verdict')}
old_path=OUT/'source_record.json'
if old_path.exists(): require(json.loads(old_path.read_text())==source_record,'Resume blocked: source lineage changed.')
else: atomic_json(old_path,source_record)

for p in [N45/'mask_lock.json',N45/'source_record.json',N46/'source_record.json',N46/'restorer_pre_registration.json',N46/'v8_6_verdict.json',N46/'audit/output_completion.json',N46/'audit/learned_model_inventory.csv',N50/'notebook50_verdict.json',N50/'m5_map_inventory.csv']:
    dst=INPUT/(p.parent.name+'__'+p.name); shutil.copy2(p,dst)
print(json.dumps(source_record,indent=2)); print('Pages:',regions.page.nunique(),'Source groups:',regions.source_group.nunique(),'Regions:',len(regions))


## 3. Exact Notebook 46R6 TinyUNet and held-out checkpoint resolver


In [ ]:
class TinyUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.e1=nn.Sequential(nn.Conv2d(2,32,3,padding=1),nn.ReLU(),nn.Conv2d(32,32,3,padding=1),nn.ReLU())
        self.p=nn.MaxPool2d(2)
        self.e2=nn.Sequential(nn.Conv2d(32,64,3,padding=1),nn.ReLU(),nn.Conv2d(64,64,3,padding=1),nn.ReLU())
        self.u=nn.ConvTranspose2d(64,32,2,2)
        self.d=nn.Sequential(nn.Conv2d(64,32,3,padding=1),nn.ReLU(),nn.Conv2d(32,1,1),nn.Sigmoid())
    def forward(self,x):
        a=self.e1(x); b=self.e2(self.p(a)); return self.d(torch.cat([self.u(b),a],1))

def load_n46_model(path):
    model=TinyUNet().to(DEVICE); state=torch.load(path,map_location=DEVICE); model.load_state_dict(state); model.eval()
    with torch.no_grad(): y=model(torch.zeros((1,2,PATCH,PATCH),dtype=torch.float32,device=DEVICE))
    require(tuple(y.shape)==(1,1,PATCH,PATCH) and torch.isfinite(y).all(),f'Checkpoint smoke test failed: {path}')
    return model

mi=pd.read_csv(N46/'audit/learned_model_inventory.csv')
required={'test_source_group','checkpoint','sha256'}; require(required.issubset(mi.columns),f'Notebook46 learned model inventory missing {sorted(required-set(mi.columns))}')
checkpoint_by_group={}
for _,r in mi.iterrows():
    g=str(r.test_source_group); p=Path(str(r.checkpoint))
    if not p.exists():
        # Safe recovery if the Notebook46 run was moved but models stayed in its models folder.
        alt=N46/'models'/p.name
        require(alt.exists(),f'Missing Notebook46 checkpoint for {g}: {p}'); p=alt
    require(sha(p)==str(r.sha256),f'Notebook46 checkpoint hash mismatch for {g}: {p}')
    checkpoint_by_group[g]=p
missing_groups=sorted(set(regions.source_group)-set(checkpoint_by_group)); require(not missing_groups,f'No held-out Notebook46 model for source groups: {missing_groups}')
print('Verified held-out TinyUNet checkpoints:',len(checkpoint_by_group))


## 4. Page inputs, frozen M3/M4 masks and Notebook 46 whole-page bases


In [ ]:
source_v82=Path(source45['source_v82'])
clean_dir=ROOT/'source'/'clean'
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
clean_map={p.stem:p for p in clean_dir.iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED}
pages=sorted(regions.page.unique())
page_obs={}; page_base={}; page_auth={}; page_protected={}; page_masks={}

def union_arm(page,arm):
    arr=None
    for _,r in regions[regions.page==page].iterrows():
        p=N45/'masks'/page/str(r.region_id).replace(':','_')/f'{arm}.png'; m=cv2.imread(str(p),0)
        require(m is not None,f'Missing/unreadable Notebook45 mask {p}'); b=m>0; arr=b.copy() if arr is None else (arr|b)
    return arr

for page in pages:
    obs=cv2.imread(str(source_v82/'whole_pages'/page/'observed.png'),0)
    base=cv2.imread(str(N46/'whole_page_candidates'/page/'v7_8.png'),0)
    protected=cv2.imread(str(N46/'whole_page_candidates'/page/'protected_print.png'),0)
    if base is None: base=cv2.imread(str(source_v82/'whole_pages'/page/'v7_8.png'),0)
    require(obs is not None and base is not None and protected is not None,f'{page}: missing observed/base/protected source page.')
    v75=Path(source45['v75']); status=pd.read_csv(v75/'tables/cache_status.csv'); cache={str(r.page):Path(r.cache) for _,r in status.iterrows()}
    require(page in cache,f'{page}: missing V7.5 cache mapping.')
    with np.load(cache[page]) as z: auth=z['mask']>0
    require(obs.shape==base.shape==protected.shape==auth.shape,f'{page}: page source shape mismatch.')
    require(np.array_equal(base[~auth],obs[~auth]),f'{page}: V7.8 outside-authority drift.')
    page_obs[page]=obs; page_base[page]=base; page_auth[page]=auth; page_protected[page]=protected>0
    page_masks[page]={'M3':union_arm(page,'M3'),'M4_DIAGNOSTIC':union_arm(page,'M4')}
print('Loaded page bases and frozen M3/M4 unions:',len(pages))


## 5. Resolve Notebook 50 SAFE_REMOVE maps and construct M5-REMOVE


In [ ]:
inv50=pd.read_csv(N50/'m5_map_inventory.csv'); require('page' in inv50.columns,'Notebook50 m5_map_inventory.csv lacks page.')
inv50['page']=inv50.page.astype(str)

def candidates_from_inventory(page):
    q=inv50[inv50.page==page]; hits=[]
    for c in q.columns:
        if any(k in c.lower() for k in ['remove','map','mask','path','file']):
            for v in q[c].dropna().astype(str):
                p=Path(v)
                if p.exists() and p.suffix.lower() in SUPPORTED and ('remove' in p.name.lower() or 'safe' in p.name.lower()): hits.append(p)
    return hits

def resolve_safe_remove(page):
    hits=candidates_from_inventory(page)
    patterns=[f'**/{page}/safe_remove.png',f'**/{page}/SAFE_REMOVE.png',f'**/{page}*safe_remove*.png',f'**/{page}*SAFE_REMOVE*.png']
    for pat in patterns: hits.extend(N50.glob(pat))
    # unique readable candidates; accept only a map that is a subset of M3.
    seen=[]
    for p in hits:
        p=Path(p)
        if str(p) in seen: continue
        seen.append(str(p)); a=cv2.imread(str(p),0)
        if a is None or a.shape!=page_masks[page]['M3'].shape: continue
        b=a>0
        if not np.any(b & ~page_masks[page]['M3']): return b,p
    raise RuntimeError(f'{page}: could not resolve a Notebook50 SAFE_REMOVE map that is a subset of M3. Set N50_RUN_OVERRIDE if needed.')

mask_rows=[]
for page in pages:
    sr,path=resolve_safe_remove(page); m3=page_masks[page]['M3']; m5=m3 & ~sr
    page_masks[page]['SAFE_REMOVE']=sr; page_masks[page]['M5_REMOVE']=m5
    mask_rows.append({'page':page,'source_group':str(regions.loc[regions.page==page,'source_group'].iloc[0]),'M3_pixels':int(m3.sum()),'SAFE_REMOVE_pixels':int(sr.sum()),'M5_REMOVE_pixels':int(m5.sum()),'M4_pixels':int(page_masks[page]['M4_DIAGNOSTIC'].sum()),'safe_remove_subset_m3':bool(not np.any(sr&~m3)),'safe_remove_path':str(path),'safe_remove_sha256':sha(path)})
mask_df=pd.DataFrame(mask_rows); atomic_csv(mask_df,TABLES/'mask_arm_inventory.csv')
require(mask_df.safe_remove_subset_m3.all(),'SAFE_REMOVE subset integrity failed.')
print(mask_df[['M3_pixels','SAFE_REMOVE_pixels','M5_REMOVE_pixels','M4_pixels']].sum())


## 6. Exact reconstruction functions


In [ ]:
def reconstruct_classical(obs,mask,method):
    m=(mask>0).astype(np.uint8)*255
    if not m.any(): return obs.copy()
    if method=='R2_telea': cand=cv2.inpaint(obs,m,3,cv2.INPAINT_TELEA)
    elif method=='R1_biharmonic': cand=np.clip(sk_inpaint.inpaint_biharmonic(obs.astype(np.float32)/255.,m>0,channel_axis=None)*255,0,255).astype(np.uint8)
    else: raise ValueError(method)
    out=obs.copy(); out[m>0]=cand[m>0]; require(np.array_equal(out[m==0],obs[m==0]),'Classical restorer outside-mask drift.'); return out

def centred_patch_geometry(shape,y0,y1,x0,x1,size=PATCH):
    h,w=map(int,shape[:2]); cy=(int(y0)+int(y1))//2; cx=(int(x0)+int(x1))//2; half=size//2
    ny0=cy-half; nx0=cx-half; ny1=ny0+size; nx1=nx0+size
    py0=max(0,ny0); px0=max(0,nx0); py1=min(h,ny1); px1=min(w,nx1)
    qy0=py0-ny0; qx0=px0-nx0; qy1=qy0+(py1-py0); qx1=qx0+(px1-px0)
    return {'page':(slice(py0,py1),slice(px0,px1)),'patch':(slice(qy0,qy1),slice(qx0,qx1))}

def crop_pad(a,y0,y1,x0,x1,size=PATCH,pad_mode='reflect'):
    g=centred_patch_geometry(a.shape,y0,y1,x0,x1,size); core=a[g['page']]; py0=g['patch'][0].start; px0=g['patch'][1].start; py1=size-g['patch'][0].stop; px1=size-g['patch'][1].stop
    mode=pad_mode
    if mode=='reflect' and (core.shape[0]<2 or core.shape[1]<2): mode='edge'
    if mode=='constant': out=np.pad(core,((py0,py1),(px0,px1)),mode='constant',constant_values=0)
    else: out=np.pad(core,((py0,py1),(px0,px1)),mode=mode)
    require(out.shape==(size,size),f'crop_pad shape {out.shape}'); return out

def run_learned(model,obs,mask,r):
    y0,y1,x0,x1=map(int,[r.y0,r.y1,r.x0,r.x1]); crop=crop_pad(obs,y0,y1,x0,x1); mc=crop_pad(mask.astype(np.uint8),y0,y1,x0,x1,pad_mode='constant')>0
    paper=int(np.median(crop[crop>200])) if np.any(crop>200) else 255; corrupted=crop.copy(); corrupted[mc]=paper
    x=np.stack([corrupted.astype(np.float32)/255.,mc.astype(np.float32)]).astype(np.float32)
    with torch.no_grad(): pred=(model(torch.from_numpy(x)[None].to(DEVICE))[0,0].cpu().numpy()*255).clip(0,255).astype(np.uint8)
    return pred

def place_prediction(original,pred_patch,full_mask,r):
    g=centred_patch_geometry(original.shape,int(r.y0),int(r.y1),int(r.x0),int(r.x1),PATCH); out=original.copy(); lm=full_mask[g['page']]; lp=pred_patch[g['patch']]; lo=out[g['page']]
    require(lo.shape==lm.shape==lp.shape,f'Placement mismatch {r.region_id}'); lo[lm]=lp[lm]; out[g['page']]=lo
    require(np.array_equal(out[~full_mask],original[~full_mask]),f'{r.region_id}: learned outside-mask drift'); return out


## 7. Candidate construction, page-checkpointed

M3 and M4 are copied from Notebook 46R6's locked whole-page candidates. M5-REMOVE is the only newly constructed arm. For M5, each Notebook 45 region receives the page-global M5 mask intersected with that region's M3 footprint, preserving the same region geometry and held-out source-group model assignment used by Notebook 46R6.


In [ ]:
# Build lookup for frozen Notebook46 M3/M4 candidates.
c46=pd.read_csv(N46/'tables/whole_page_candidate_inventory.csv')
c46['page']=c46.page.astype(str)

def frozen46(page,arm,rest):
    q=c46[(c46.page==page)&(c46.arm==arm)&(c46.restorer==rest)]
    require(len(q)==1,f'{page}/{arm}/{rest}: expected one Notebook46 whole-page candidate, got {len(q)}')
    p=Path(str(q.iloc[0].candidate)); require(p.exists(),f'Missing Notebook46 candidate {p}'); return p

def checkpoint_path(page): return CHECK/f'{page}.json'
def valid_checkpoint(page,contract):
    p=checkpoint_path(page)
    if not p.exists(): return False
    try:
        x=json.loads(p.read_text());
        if x.get('contract')!=contract or x.get('complete') is not True: return False
        return all(Path(r['path']).exists() and sha(Path(r['path']))==r['sha256'] for r in x['outputs'])
    except Exception: return False

contract=hashlib.sha256(json.dumps({'prereg':PREREG,'n45_lock':sha(N45/'mask_lock.json'),'n46_verdict':sha(N46/'v8_6_verdict.json'),'n46_pre':sha(N46/'restorer_pre_registration.json'),'n46_models':sha(N46/'audit/learned_model_inventory.csv'),'n50_verdict':sha(N50/'notebook50_verdict.json'),'n50_inventory':sha(N50/'m5_map_inventory.csv')},sort_keys=True).encode()).hexdigest()
status=[]
for page in pages:
    if valid_checkpoint(page,contract): status.append({'page':page,'status':'reused'}); continue
    obs=page_obs[page]; auth=page_auth[page]; protected=page_protected[page]; page_dir=CAND/page; page_dir.mkdir(parents=True,exist_ok=True); outrows=[]
    # Reuse frozen M3 / M4 whole-page candidates exactly.
    for arm46,arm51 in [('M3','M3'),('M4','M4_DIAGNOSTIC')]:
        for rest in RESTORERS:
            src=frozen46(page,arm46,rest); dst=page_dir/f'{arm51}_{rest}.png'; shutil.copy2(src,dst); im=cv2.imread(str(dst),0)
            require(im is not None and im.shape==obs.shape,f'{page}/{arm51}/{rest}: unreadable frozen candidate'); require(np.array_equal(im[~auth],obs[~auth]),f'{page}/{arm51}/{rest}: frozen outside-mask drift')
            outrows.append({'arm':arm51,'restorer':rest,'path':str(dst),'sha256':sha(dst),'source':'Notebook46R6_frozen'})
    # Construct M5-REMOVE from the same base/protected whole-page contract.
    m5=page_masks[page]['M5_REMOVE']; candidate_base=page_base[page].copy(); candidate_base[protected]=obs[protected]
    group=str(regions.loc[regions.page==page,'source_group'].iloc[0]); model=load_n46_model(checkpoint_by_group[group])
    pg=regions[regions.page==page]
    for rest in RESTORERS:
        candidate=candidate_base.copy()
        for _,r in pg.iterrows():
            # Restrict page-global M5 to the region's original M3 footprint. This preserves Notebook46 regional ownership/placement semantics.
            mp=N45/'masks'/page/str(r.region_id).replace(':','_')/'M3.png'; rm=cv2.imread(str(mp),0); require(rm is not None and rm.shape==obs.shape,f'Invalid regional M3 mask {mp}')
            regional=m5 & (rm>0)
            if not regional.any(): continue
            if rest=='R3_learned_unet': pred=run_learned(model,obs,regional,r); recon=place_prediction(obs,pred,regional,r)
            else: recon=reconstruct_classical(obs,regional,rest)
            candidate[regional]=recon[regional]
        require(np.array_equal(candidate[~auth],obs[~auth]),f'{page}/M5/{rest}: outside-authority drift')
        require(np.array_equal(candidate[protected & ~m5],obs[protected & ~m5]),f'{page}/M5/{rest}: protected non-residual drift')
        dst=page_dir/f'M5_REMOVE_{rest}.png'; tmp=dst.with_suffix('.tmp.png'); require(cv2.imwrite(str(tmp),candidate),f'Cannot write {tmp}'); tmp.replace(dst)
        outrows.append({'arm':'M5_REMOVE','restorer':rest,'path':str(dst),'sha256':sha(dst),'source':'Notebook51R2_new'})
    atomic_json(checkpoint_path(page),{'version':'51R2-page-v1','page':page,'complete':True,'contract':contract,'outputs':outrows,'created_utc':datetime.now(timezone.utc).isoformat()}); status.append({'page':page,'status':'computed'})
    del model; gc.collect(); torch.cuda.empty_cache()
atomic_csv(pd.DataFrame(status),AUDIT/'page_completion.csv'); print(pd.DataFrame(status).status.value_counts())


## 8. Candidate lock before clean truth


In [ ]:
rows=[]
for page in pages:
    sg=str(regions.loc[regions.page==page,'source_group'].iloc[0])
    for arm in ARMS:
        for rest in RESTORERS:
            p=CAND/page/f'{arm}_{rest}.png'; require(p.exists(),f'Missing candidate {p}'); rows.append({'page':page,'source_group':sg,'arm':arm,'restorer':rest,'candidate':str(p),'sha256':sha(p),'array_sha256':array_hash(cv2.imread(str(p),0))})
candidate_inventory=pd.DataFrame(rows).sort_values(['page','arm','restorer']).reset_index(drop=True); atomic_csv(candidate_inventory,LOCK/'candidate_inventory.csv')
lock_id=hashlib.sha256(candidate_inventory.to_csv(index=False).encode()).hexdigest(); atomic_json(LOCK/'candidate_lock.json',{'version':'51R2-candidate-lock-v1','lock_id':lock_id,'contract':contract,'rows':len(candidate_inventory),'truth_loaded':False,'created_utc':datetime.now(timezone.utc).isoformat()})
print('CANDIDATE LOCKED:',lock_id,'rows:',len(candidate_inventory))


## 9. Unlock clean truth and score fixed authoritative-mask region


In [ ]:
clean_map={p.stem:p for p in (ROOT/'source'/'clean').iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED}
require(set(pages).issubset(clean_map),f'Missing clean pages after candidate lock: {sorted(set(pages)-set(clean_map))}')

def ink(im): return cv2.adaptiveThreshold(im,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
def edges(im): return cv2.Canny(im,70,170)>0
def f1(ref,pred,reg,tol=1):
    ref=ref&reg; pred=pred&reg
    if not ref.any(): return np.nan,np.nan,np.nan
    k=np.ones((2*tol+1,2*tol+1),np.uint8); p=np.sum(pred&(cv2.dilate(ref.astype(np.uint8),k)>0))/max(pred.sum(),1); r=np.sum(ref&(cv2.dilate(pred.astype(np.uint8),k)>0))/ref.sum(); return float(p),float(r),float(2*p*r/max(p+r,1e-12))

score_rows=[]
for _,row in candidate_inventory.iterrows():
    page=row.page; clean=cv2.imread(str(clean_map[page]),0); pred=cv2.imread(str(row.candidate),0); obs=page_obs[page]; auth=page_auth[page]
    require(clean is not None and pred is not None and clean.shape==pred.shape==auth.shape,f'{page}: scoring shape/read failure')
    diff=np.abs(clean.astype(np.float32)-pred.astype(np.float32)); _,sm=ssim(clean,pred,data_range=255,full=True); ip,ir,if1=f1(ink(clean),ink(pred),auth); ep,er,ef1=f1(edges(clean),edges(pred),auth)
    score_rows.append({'page':page,'source_group':row.source_group,'arm':row.arm,'restorer':row.restorer,'authoritative_pixels':int(auth.sum()),'mae':float(diff[auth].mean()),'ssim':float(sm[auth].mean()),'ink_precision':ip,'ink_recall':ir,'ink_f1':if1,'edge_f1':ef1,'false_inserted_ink':int((ink(pred)&~ink(clean)&auth).sum()),'missing_ink':int((ink(clean)&~ink(pred)&auth).sum()),'outside_mask_drift':int(np.sum(pred[~auth]!=obs[~auth]))})
metrics=pd.DataFrame(score_rows); require((metrics.outside_mask_drift==0).all(),'Outside-authoritative-mask drift detected in scored candidates.'); atomic_csv(metrics,TABLES/'candidate_metrics.csv')
summary=metrics.groupby(['arm','restorer'],as_index=False).agg(pages=('page','nunique'),mae=('mae','mean'),ssim=('ssim','mean'),ink_f1=('ink_f1','mean'),edge_f1=('edge_f1','mean'),false_inserted_ink=('false_inserted_ink','sum'),missing_ink=('missing_ink','sum'),outside_mask_drift=('outside_mask_drift','sum'))
atomic_csv(summary,TABLES/'candidate_summary.csv'); display(summary)


## 10. Paired page-clustered M5-REMOVE vs M3 inference


In [ ]:
def benefit(metric,m5,m3): return (m3-m5) if metric in ['mae','false_inserted_ink','missing_ink'] else (m5-m3)
def boot_ci(v,n=BOOT,seed=SEED):
    v=np.asarray(v,float); v=v[np.isfinite(v)]
    if not len(v): return np.nan,np.nan,np.nan
    rng=np.random.default_rng(seed); b=np.array([rng.choice(v,len(v),replace=True).mean() for _ in range(n)]); return float(v.mean()),float(np.quantile(b,.025)),float(np.quantile(b,.975))

paired=[]
for rest,g in metrics.groupby('restorer'):
    for metric in ['mae','ssim','ink_f1','edge_f1','false_inserted_ink','missing_ink']:
        p=g.pivot(index='page',columns='arm',values=metric); d=benefit(metric,p.M5_REMOVE,p.M3); mean,lo,hi=boot_ci(d)
        paired.append({'restorer':rest,'metric':metric,'benefit_mean':mean,'ci95_low':lo,'ci95_high':hi,'wins':int((d>0).sum()),'losses':int((d<0).sum()),'ties':int((d==0).sum()),'pages':int(d.notna().sum())})
paired_df=pd.DataFrame(paired); atomic_csv(paired_df,TABLES/'paired_m5_remove_vs_m3_bootstrap.csv'); display(paired_df)

source=[]
for (sg,rest),g in metrics.groupby(['source_group','restorer']):
    for metric in ['mae','ssim','ink_f1','edge_f1','false_inserted_ink','missing_ink']:
        p=g.pivot(index='page',columns='arm',values=metric); d=benefit(metric,p.M5_REMOVE,p.M3)
        source.append({'source_group':sg,'restorer':rest,'metric':metric,'pages':int(len(d)),'benefit_mean':float(d.mean()),'wins':int((d>0).sum()),'losses':int((d<0).sum()),'ties':int((d==0).sum())})
source_df=pd.DataFrame(source); atomic_csv(source_df,TABLES/'m5_remove_vs_m3_by_source_group.csv')


## 11. Restorer-gap contraction


In [ ]:
gap_rows=[]
for (page,arm),g in metrics.groupby(['page','arm']):
    row={'page':page,'source_group':g.source_group.iloc[0],'arm':arm}
    for metric in ['mae','ink_f1','edge_f1','false_inserted_ink','missing_ink']:
        v=g[metric].to_numpy(float); row[f'{metric}_range']=float(np.nanmax(v)-np.nanmin(v)); row[f'{metric}_sd']=float(np.nanstd(v))
    gap_rows.append(row)
gaps=pd.DataFrame(gap_rows); atomic_csv(gaps,TABLES/'restorer_gap_by_page.csv')
contract_rows=[]
for metric in ['mae','ink_f1','edge_f1','false_inserted_ink','missing_ink']:
    for stat in ['range','sd']:
        p=gaps.pivot(index='page',columns='arm',values=f'{metric}_{stat}'); d=p.M3-p.M5_REMOVE; mean,lo,hi=boot_ci(d)
        contract_rows.append({'metric':metric,'gap_stat':stat,'contraction_mean':mean,'ci95_low':lo,'ci95_high':hi,'pages_contracting':int((d>0).sum()),'pages_expanding':int((d<0).sum()),'ties':int((d==0).sum())})
contraction=pd.DataFrame(contract_rows); atomic_csv(contraction,TABLES/'restorer_gap_contraction.csv'); display(contraction)


## 12. Diagnostic M4 positioning


In [ ]:
# M4 is not a promotion target. Report distance from M3/M5 to the diagnostic reference only.
rows=[]
for rest,g in metrics.groupby('restorer'):
    for metric in ['mae','ssim','ink_f1','edge_f1','false_inserted_ink','missing_ink']:
        p=g.pivot(index='page',columns='arm',values=metric)
        higher=metric in ['ssim','ink_f1','edge_f1']
        # Positive values mean the first mask arm is better than the second.
        m5_m4=(p.M5_REMOVE-p.M4_DIAGNOSTIC)*(1 if higher else -1)
        m3_m4=(p.M3-p.M4_DIAGNOSTIC)*(1 if higher else -1)
        rows.append({'restorer':rest,'metric':metric,'M5_minus_M4_benefit_mean':float(m5_m4.mean()),'M3_minus_M4_benefit_mean':float(m3_m4.mean())})
atomic_csv(pd.DataFrame(rows),TABLES/'diagnostic_m4_positioning.csv')


## 13. Visual audit queues


In [ ]:
queue=[]
for rest,g in metrics.groupby('restorer'):
    p=g.set_index(['page','arm'])
    for page in pages:
        a=p.loc[(page,'M3')]; b=p.loc[(page,'M5_REMOVE')]
        queue.append({'page':page,'source_group':a.source_group,'restorer':rest,'edge_f1_benefit':b.edge_f1-a.edge_f1,'ink_f1_benefit':b.ink_f1-a.ink_f1,'mae_benefit':a.mae-b.mae,'false_inserted_benefit':a.false_inserted_ink-b.false_inserted_ink,'missing_ink_benefit':a.missing_ink-b.missing_ink})
queue=pd.DataFrame(queue); atomic_csv(queue,TABLES/'visual_audit_queue.csv')
for rest in RESTORERS:
    for label,q in [('best',queue[queue.restorer==rest].sort_values(['edge_f1_benefit','ink_f1_benefit'],ascending=False).head(10)),('worst',queue[queue.restorer==rest].sort_values(['edge_f1_benefit','ink_f1_benefit']).head(10))]:
        if q.empty: continue
        fig,axes=plt.subplots(len(q),3,figsize=(15,5*len(q)),squeeze=False)
        for i,row in enumerate(q.itertuples()):
            for j,(arm,title) in enumerate([('M3','M3'),('M5_REMOVE','M5-REMOVE'),('M4_DIAGNOSTIC','M4 diagnostic')]):
                im=cv2.imread(str(CAND/row.page/f'{arm}_{rest}.png'),0); axes[i,j].imshow(im,cmap='gray',vmin=0,vmax=255); axes[i,j].set_title(f'{row.page} {title}'); axes[i,j].axis('off')
        plt.tight_layout(); plt.savefig(VIS/f'{rest}_{label}_whole_pages.png',dpi=120,bbox_inches='tight'); plt.close(fig)
print('Visual audit:',VIS)


## 14. Fail-closed interpretation


In [ ]:
# Predeclared conservative research interpretation. Positive CI means M5 benefit.
struct=paired_df[paired_df.metric.isin(['ink_f1','edge_f1'])].copy()
positive_restorers=sorted(set(struct.loc[struct.ci95_low>0,'restorer']))
harmful=struct[struct.ci95_high<0].copy()
robust_contraction=contraction[contraction.ci95_low>0].copy()
if len(harmful): verdict='M5_REMOVE_RECONSTRUCTION_REGRESSION'
elif len(positive_restorers)>=2 and len(robust_contraction): verdict='M5_REMOVE_CAUSAL_GAIN_WITH_RESTORER_GAP_CONTRACTION'
elif len(positive_restorers)>=2: verdict='M5_REMOVE_CAUSAL_GAIN_WITHOUT_ROBUST_GAP_CONTRACTION'
else: verdict='M5_REMOVE_CAUSAL_GAIN_NOT_DEMONSTRATED'
record={'version':'8.7-notebook51R2','valid_run':True,'verdict':verdict,'development_only':True,'promotion_authority':False,'positive_structural_restorers':positive_restorers,'structural_harm_rows':harmful.to_dict('records'),'robust_gap_contraction_endpoints':robust_contraction[['metric','gap_stat','contraction_mean','ci95_low','ci95_high']].to_dict('records'),'m5_policy':'REMOVE-only','safe_protect_intervention':False,'m4_role':'diagnostic only'}
atomic_json(OUT/'notebook51_verdict.json',record); print(json.dumps(record,indent=2))


## 15. Identity and Final Upload Package


In [ ]:
environment={'python':sys.version,'torch':torch.__version__,'cuda':torch.version.cuda,'gpu':torch.cuda.get_device_name(0),'opencv':cv2.__version__,'numpy':np.__version__,'pandas':pd.__version__,'platform':platform.platform()}
atomic_json(OUT/'software_environment.json',environment)
identity={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'8.7-notebook51R2','candidate_lock_id':lock_id,'contract':contract,'source_record':source_record,'mask_arms':ARMS,'restorers':RESTORERS,'heldout_unet_checkpoints':{g:{'path':str(p),'sha256':sha(p)} for g,p in checkpoint_by_group.items()},'pages':len(pages),'source_groups':int(regions.source_group.nunique()),'development_only':True,'promotion_authority':False,'software':environment}
atomic_json(OUT/'experiment_identity.json',identity)
lines=['V8.7 NOTEBOOK 51R2 - FROZEN M3 / M5-REMOVE / DIAGNOSTIC-M4 CROSSOVER','',f'Run: {OUT}',f'Pages: {len(pages)}',f'Source groups: {regions.source_group.nunique()}',f'Verdict: {verdict}','',f'Positive structural restorers: {positive_restorers}',f'Robust restorer-gap contraction endpoints: {len(robust_contraction)}','','M5 contains SAFE_REMOVE only. SAFE_PROTECT is not applied. M4 is diagnostic only. Clean truth was loaded only after the candidate lock. No production authority.']
(OUT/'executive_summary.txt').write_text(chr(10).join(lines))
required=[OUT/'executive_summary.txt',OUT/'notebook51_verdict.json',OUT/'experiment_identity.json',OUT/'source_record.json',OUT/'01_pre_registration.json',LOCK/'candidate_lock.json',TABLES/'mask_arm_inventory.csv',TABLES/'candidate_summary.csv',TABLES/'candidate_metrics.csv',TABLES/'paired_m5_remove_vs_m3_bootstrap.csv',TABLES/'m5_remove_vs_m3_by_source_group.csv',TABLES/'restorer_gap_contraction.csv',TABLES/'diagnostic_m4_positioning.csv',TABLES/'visual_audit_queue.csv',AUDIT/'page_completion.csv']
for pth in required: require(pth.exists(),f'Missing upload artefact {pth}')
for pth in required: shutil.copy2(pth,UPLOAD/pth.name)
(UPLOAD/'README.txt').write_text('Notebook 51R2 curated review package. Heavy candidate images and checkpoints remain in the durable run directory and are referenced by the identity/lock records.'+chr(10))
manifest=[]
for pth in sorted(UPLOAD.iterdir()):
    if pth.is_file(): manifest.append({'file':pth.name,'bytes':pth.stat().st_size,'sha256':sha(pth)})
atomic_csv(pd.DataFrame(manifest),UPLOAD/'upload_manifest.csv')
zip_path=UPLOAD/'notebook51R2_upload_bundle.zip'; tmp=UPLOAD/'notebook51R2_upload_bundle.tmp.zip'
with zipfile.ZipFile(tmp,'w',zipfile.ZIP_DEFLATED) as z:
    for pth in sorted(UPLOAD.iterdir()):
        if pth.is_file() and pth not in {zip_path,tmp}: z.write(pth,pth.name)
tmp.replace(zip_path)
print(chr(10).join(lines)); print('Bundle:',zip_path)


## Outputs to inspect first
1. `executive_summary.txt`
2. `notebook51_verdict.json`
3. `05_tables/paired_m5_remove_vs_m3_bootstrap.csv`
4. `05_tables/m5_remove_vs_m3_by_source_group.csv`
5. `05_tables/restorer_gap_contraction.csv`
6. `05_tables/candidate_summary.csv`
7. `06_visual_audit/`
8. `upload/notebook51R2_upload_bundle.zip`

If source paths have moved, set the explicit run override at the top. Do not weaken an integrity check merely to make the notebook continue.
